# Session 6, Cluster the days

> **A bike-share operator has 731 days of data and no labels.** What *kinds* of day are there, and how should the operator run each kind differently? You'll cluster the days, **name** the groups, and decide **one action**. Claude writes the code; the naming and the decision are yours, and that is what's graded.

In [ ]:
# --- Load the data -------------------------------------------------
DATA_URL = ""   # paste a hosted CSV link, or leave blank to upload bike_days.csv
import pandas as pd
if DATA_URL:
    df = pd.read_csv(DATA_URL)
else:
    try:
        from google.colab import files
        print("Upload bike_days.csv ...")
        df = pd.read_csv(next(iter(files.upload())))
    except Exception:
        df = pd.read_csv("data/bike_days.csv")
df.head()

## The columns you'll use

One row per day. Useful features: `temp` (normalised temperature), `hum` (humidity), `windspeed`, `casual` (casual riders), `registered` (member riders), plus `workingday` and `weathersit` for context. There is **no** column telling you the 'type' of day. That is what you're discovering.

## Step 1, Standardise the features  🎯

Predict first: *how many kinds of day do you expect?* Then scale, because the features are on very different ranges (`temp` 0–1 vs `registered` in the thousands).

> Ask Claude: *"Standardise temp, hum, windspeed, casual and registered from df."*

In [ ]:
from sklearn.preprocessing import StandardScaler
feats = ['temp', 'hum', 'windspeed', 'casual', 'registered']
Z = StandardScaler().fit_transform(df[feats])

## Step 2, Choose k (elbow + silhouette)

> Ask Claude: *"Show the elbow (inertia, k=1..8) and silhouette (k=2..8) for Z."* Then **pick a k**. There is no true answer, so pick one you can act on.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
for k in range(2, 8):
    lab = KMeans(n_clusters=k, n_init=10, random_state=7).fit_predict(Z)
    print(f'k={k}  silhouette={silhouette_score(Z, lab):.3f}')

## Step 3, Cluster, then look at what each group is like

> Ask Claude: *"Fit k-means with the k I chose, add the labels to df, and show the average temp, casual, registered, cnt and workingday for each cluster."*

In [ ]:
k = 4   # <- your choice from Step 2
df['cluster'] = KMeans(n_clusters=k, n_init=10, random_state=7).fit_predict(Z)
df.groupby('cluster')[['temp', 'casual', 'registered', 'cnt', 'workingday']].mean().round(2)

## Step 4, Name the day-types  ✍️

Look at each cluster's profile and give it a **plain-English name** a colleague would understand. For example: *warm leisure weekends*, *warm commuter weekdays*, *cold quiet days*.

If two clusters look almost the same, that's a signal your k may be too high, so go back and try one fewer.

In [ ]:
# Write your names here, one per cluster:
# 0 = ...
# 1 = ...
# 2 = ...
# 3 = ...

## Step 5, Decide

Pick **one** day-type and state **one thing** the operator should do differently for it: staffing, how many bikes to place where, or pricing. One or two sentences, tied to what that cluster actually looks like.

In [ ]:
# Your recommendation to the operator:
#

## Before you leave

1. Why did we standardise before clustering? What would have happened if we hadn't?
2. Your clusters have no 'correct' answer. How would you convince a manager your grouping is *useful*?
3. Which single feature seems to separate the day-types most? (Hint: look at the cluster averages.)

**Next week:** we do this for *people* instead of days: cluster customers into segments, turn each into a persona, and market to them. It's exactly your group-project job on the bank data.